In [ ]:
import pandas as pd
import glob

# Step 1: Specify the directory containing the CSV files
file_path_pattern = r'C:\Users\Administrator\Desktop\Apr_Match\*.csv'  # Update this path

# Step 2: Get a list of all CSV files in the directory
csv_files = glob.glob(file_path_pattern)

# Step 3: Read each CSV file into a DataFrame and store them in a list
dataframes = []
for file in csv_files:
    df = pd.read_csv(file)
    dataframes.append(df)
    print(f"Loaded {file} with {len(df)} rows.")

# Step 4: Access data from each DataFrame
for i, df in enumerate(dataframes):
    print(f"\nData from file {csv_files[i]}:")
    
df.head() 

In [ ]:
import pandas as pd

# Step 1: Load the clubbed file
clubbed_file_path = r'C:\Users\Administrator\Desktop\Apr_Match\clubed.csv'
clubbed_data = pd.read_csv(clubbed_file_path)

# Step 2: Load the data 227 file
data_227_file_path = r'C:\Users\Administrator\Desktop\Apr_Match\data_227.csv'
data_227 = pd.read_csv(data_227_file_path)

# Remove leading/trailing spaces from column names
clubbed_data.columns = clubbed_data.columns.str.strip()
data_227.columns = data_227.columns.str.strip()

# Print column names for debugging
print("Clubed Data Columns:")
print(clubbed_data.columns)

print("\nData 227 Columns:")
print(data_227.columns)

# Step 3: Merge the two DataFrames on the specified columns
merged_data = pd.merge(
    clubbed_data,
    data_227,
    how='inner',
    left_on=['ManagerID', 'Symbol', 'ExpiryDate', 'Option Type', 'Strike Price', 'Trade_Date'],  # Adjust as necessary
    right_on=['CLIENT_ID', 'SCRIP_SYMBOL', 'SCRIP_NAME', 'TRADE_DATE1']  # Corrected keys
)

# Step 4: Save the merged DataFrame to a new CSV file
output_file_path = r'C:\Users\Administrator\Desktop\matched_output.csv'
merged_data.to_csv(output_file_path, index=False)

print(f"Matching rows saved to {output_file_path}.")


In [ ]:
import pandas as pd
import os

# Define the file paths
clubed_path = r'C:\Users\Administrator\Desktop\Clubed.csv'
data227_path = r'C:\Users\Administrator\Desktop\Data_227.csv'

# Check if the files exist
if not os.path.isfile(clubed_path):
    print(f"File not found: {clubed_path}")
if not os.path.isfile(data227_path):
    print(f"File not found: {data227_path}")

# Load the CSV files into DataFrames if they exist
if os.path.isfile(clubed_path) and os.path.isfile(data227_path):
    df_clubed = pd.read_csv(clubed_path)
    df_data227 = pd.read_csv(data227_path)

    # Clean column names by stripping whitespace
    df_clubed.columns = df_clubed.columns.str.strip()
    df_data227.columns = df_data227.columns.str.strip()

    # Function to create token format for Clubed Data
    def create_token_clubed(row):
        return f"{row['ManagerID']} {row['OptionType']} {row['Symbol']} {row['ExpiryDate']} {row['StrikePrice']}"

    # Function to create token format for Data227
    def create_token_data227(row):
        return f"{row['CLIENT_ID']} {row['SCRIP_SYMBOL']}"

    # Create the token columns in both DataFrames
    df_clubed['Token'] = df_clubed.apply(create_token_clubed, axis=1)
    df_data227['Token'] = df_data227.apply(create_token_data227, axis=1)

    # Debugging: Print the first few tokens
    print("Sample Tokens from Clubed DataFrame:")
    print(df_clubed['Token'].head())

    print("Sample Tokens from Data227 DataFrame:")
    print(df_data227['Token'].head())

    # Merge DataFrames on the token column
    merged_df = pd.merge(df_clubed, df_data227, on='Token', suffixes=('_clubed', '_data227'))

    # Check for any matches
    if merged_df.empty:
        print("No matches found.")
    else:
        # Select relevant columns: token, buyquantity, and sellquantity
        result_df = merged_df[['Token', 'BUY_QUANTITY', 'SALE_QUANTITY']]

        # Display the result
        print(result_df)

    # Optionally, save the result to a CSV file
    result_df.to_csv('matched_quantities.csv', index=False)


In [ ]:
import os

# Define the directory
directory = r'C:\Users\Administrator\Desktop'

# List files in the directory
files = os.listdir(directory)
print("Files in the directory:")
for file in files:
    print(file)


In [ ]:
import pandas as pd
import os

# Define the file paths
clubed_path = r'C:\Users\Administrator\Desktop\Clubed.csv'
data227_path = r'C:\Users\Administrator\Desktop\Data_227.csv'

# Check if the files exist
if not os.path.isfile(clubed_path):
    print(f"File not found: {clubed_path}")
if not os.path.isfile(data227_path):
    print(f"File not found: {data227_path}")

# Load the CSV files into DataFrames if they exist
if os.path.isfile(clubed_path) and os.path.isfile(data227_path):
    df_clubed = pd.read_csv(clubed_path)
    df_data227 = pd.read_csv(data227_path)

    # Clean column names by stripping whitespace
    df_clubed.columns = df_clubed.columns.str.strip()
    df_data227.columns = df_data227.columns.str.strip()

    # Update SCRIP_SYMBOL: remove 'EF' and 'IO'
    df_data227['SCRIP_SYMBOL'] = df_data227['SCRIP_SYMBOL'].str.replace('EF', '').str.replace('IO', '')

    # Add 'XX' to OptionType where it is blank for all symbols
    df_data227.loc[df_data227['OptionType'].isnull() | (df_data227['OptionType'] == ''), 'OptionType'] = 'XX'

    # Function to create token format for Clubed Data
    def create_token_clubed(row):
        return f"{row['ManagerID']} {row['OptionType']} {row['Symbol']} {row['ExpiryDate']} {row['StrikePrice']}"

    # Function to create token format for Data227
    def create_token_data227(row):
        return f"{row['CLIENT_ID']} {row['SCRIP_SYMBOL']}"

    # Create the token columns in both DataFrames
    df_clubed['Token'] = df_clubed.apply(create_token_clubed, axis=1)
    df_data227['Token'] = df_data227.apply(create_token_data227, axis=1)

    # Debugging: Print the first few tokens
    print("Sample Tokens from Clubed DataFrame:")
    print(df_clubed['Token'].head())

    print("Sample Tokens from Data227 DataFrame:")
    print(df_data227['Token'].head())

    # Merge DataFrames on the token column
    merged_df = pd.merge(df_clubed, df_data227, on='Token', suffixes=('_clubed', '_data227'))

    # Check for any matches
    if merged_df.empty:
        print("No matches found.")
    else:
        # Select relevant columns: token, buyquantity, and sellquantity
        result_df = merged_df[['Token', 'BUY_QUANTITY', 'SALE_QUANTITY']]

        # Display the result
        print(result_df)

    # Optionally, save the result to a CSV file
    result_df.to_csv('matched_quantities.csv', index=False)
